This is the testing notebook for me while creating the main app.py

In [2]:
#Import stuff

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np

c:\Projects\44688_80_Capstone\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating the first function in the pipeline to pull my data - this should pull the kaggle dataset and by default pull the dataset I want toherwise you can specify.

In [ ]:
def kaggle_data_pull(
    dataset_file: str, 
    dataset_url: str = "shashwatwork/dataco-smart-supply-chain-for-big-data-analysis"
) -> pd.DataFrame:
    """
    Pulls a specific dataset file from Kaggle and loads it into a Pandas DataFrame.
    """
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS, 
        dataset_url, 
        dataset_file,
        # Tell pandas to handle special characters using latin1 encoding
        pandas_kwargs={"encoding": "latin1"}
    )
    print("Dataset pulled successfully")
    return df

In [8]:
kaggle_data_pull("DataCoSupplyChainDataset.csv")

Dataset pulled successfully


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,1/15/2018 11:24,Standard Class
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180514,CASH,4,4,40.000000,399.980011,Shipping on time,0,45,Fishing,Brooklyn,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/20/2016 3:40,Standard Class
180515,DEBIT,3,2,-613.770019,395.980011,Late delivery,1,45,Fishing,Bakersfield,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/19/2016 1:34,Second Class
180516,TRANSFER,5,4,141.110001,391.980011,Late delivery,1,45,Fishing,Bristol,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/20/2016 21:00,Standard Class
180517,PAYMENT,3,4,186.229996,387.980011,Advance shipping,0,45,Fishing,Caguas,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/18/2016 20:18,Standard Class


After dataset is pulled successfully I will need to do some basic cleaning - I know my target variable for my Random Forest Regression Model. I should probably pull all column headers though first even though I have partially decided on"Days for shipping (real)" as my target variable while "Shipping Mode", "shipping date (DateOrders)", "Order City", "Order Country", "Category ID" are going into the prediction portion.

I also created a function to count the rows since we will probably need that

In [15]:
df = kaggle_data_pull("DataCoSupplyChainDataset.csv")

def get_column_headers(df: pd.DataFrame) -> list[str]:
    """
    Returns and prints all column headers in the DataFrame.
    """
    headers = list(df.columns)
    print(f"Total Columns: {len(headers)}\n")
    for idx, col in enumerate(headers, 1):
        print(f"{idx:2d}. '{col}'")
    return headers

cols = get_column_headers(df)

def get_num_rows(df: pd.DataFrame) -> int:
    """Counts and returns the total number of rows in the DataFrame."""
    num_rows = len(df)
    print(f"Total Rows: {num_rows}")
    return num_rows

total_rows = get_num_rows(df)


Dataset pulled successfully
Total Columns: 53

 1. 'Type'
 2. 'Days for shipping (real)'
 3. 'Days for shipment (scheduled)'
 4. 'Benefit per order'
 5. 'Sales per customer'
 6. 'Delivery Status'
 7. 'Late_delivery_risk'
 8. 'Category Id'
 9. 'Category Name'
10. 'Customer City'
11. 'Customer Country'
12. 'Customer Email'
13. 'Customer Fname'
14. 'Customer Id'
15. 'Customer Lname'
16. 'Customer Password'
17. 'Customer Segment'
18. 'Customer State'
19. 'Customer Street'
20. 'Customer Zipcode'
21. 'Department Id'
22. 'Department Name'
23. 'Latitude'
24. 'Longitude'
25. 'Market'
26. 'Order City'
27. 'Order Country'
28. 'Order Customer Id'
29. 'order date (DateOrders)'
30. 'Order Id'
31. 'Order Item Cardprod Id'
32. 'Order Item Discount'
33. 'Order Item Discount Rate'
34. 'Order Item Id'
35. 'Order Item Product Price'
36. 'Order Item Profit Ratio'
37. 'Order Item Quantity'
38. 'Sales'
39. 'Order Item Total'
40. 'Order Profit Per Order'
41. 'Order Region'
42. 'Order State'
43. 'Order Status'

Now that I have the stupid freaking columns and rows let me do basic data cleaning - I want to choose just the columns I want including the one I want to target.

Target:
'Days for shipping (real)'

Variables:

'Latitude'
'Longitude'
'Customer City'
'Customer Country'
'order date (DateOrders)'
'Order Item Total'
'Product Category Id'
'Shipping Mode'

Ok so now I only need to keep these target items so I will need to create a function to clean the critical data

In [25]:
df = kaggle_data_pull("DataCoSupplyChainDataset.csv")

def select_target_columns(df: pd.DataFrame, target_cols: list[str] = None) -> pd.DataFrame:
    """
    Filters the DataFrame to keep only the specified target columns.
    """
    if target_cols is None:
        target_cols = [
            'Latitude',
            'Longitude',
            'Customer City',
            'Customer Country',
            'order date (DateOrders)',
            'Order Item Total',
            'Product Category Id',
            'Shipping Mode',
            'Days for shipping (real)'  # Target variable included
        ]
    
    # Strip column headers first to prevent matching errors
    df_clean = df.copy()
    df_clean.columns = df_clean.columns.str.strip()
    
    # Filter DataFrame to keep only the target columns
    print("Dataset cleaned succesfully")
    return df_clean[target_cols]


select_target_columns(df)


Dataset pulled successfully
Dataset cleaned succesfully


,Latitude,Longitude,Customer City,Customer Country,order date (DateOrders),Order Item Total,Product Category Id,Shipping Mode,Days for shipping (real)
0,18.251453,-66.037056,Caguas,Puerto Rico,1/31/2018 22:56,314.640015,73,Standard Class,3
1,18.279451,-66.037064,Caguas,Puerto Rico,1/13/2018 12:27,311.359985,73,Standard Class,5
2,37.292233,-121.881279,San Jose,EE. UU.,1/13/2018 12:06,309.720001,73,Standard Class,4
3,34.125946,-118.291016,Los Angeles,EE. UU.,1/13/2018 11:45,304.809998,73,Standard Class,3
4,18.253769,-66.037048,Caguas,Puerto Rico,1/13/2018 11:24,298.250000,73,Standard Class,2
...,...,...,...,...,...,...,...,...,...
180514,40.640930,-73.942711,Brooklyn,EE. UU.,1/16/2016 3:40,399.980011,45,Standard Class,4
180515,35.362545,-119.018700,Bakersfield,EE. UU.,1/16/2016 1:34,395.980011,45,Second Class,3
180516,41.629959,-72.967155,Bristol,EE. UU.,1/15/2016 21:00,391.980011,45,Standard Class,5
180517,18.213350,-66.370575,Caguas,Puerto Rico,1/15/2016 20:18,387.980011,45,Standard Class,3


In [35]:

def clean_critical_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans critical columns: strips headers, fixes data types (numeric & date),
    and drops rows with missing/invalid values across ALL selected columns.
    """
    # Create an independent copy to prevent modifying the original input
    df_clean = df.copy()

    # 1. Clean column header whitespace
    df_clean.columns = df_clean.columns.str.strip()

    # 2. Define numeric columns
    numeric_cols = [
        'Latitude', 
        'Longitude', 
        'Order Item Total', 
        'Product Category Id', 
        'Days for shipping (real)'
    ]
    
    # 3. Fix wrong data types (converts bad text like 'N/A' or corrupt strings into NaN)
    for col in numeric_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

    df_clean['order date (DateOrders)'] = pd.to_datetime(
        df_clean['order date (DateOrders)'], 
        errors='coerce'
    )

    # 4. Drop any rows missing values across ALL columns in the DataFrame
    df_clean = df_clean.dropna()
    
    print(f"Rows before: {len(df)} | Rows after: {len(df_clean)}")
    print(f"Columns before: {len(df.columns)} | Columns after: {len(df_clean.columns)}")
    return df_clean


# Pipeline execution:
df = kaggle_data_pull("DataCoSupplyChainDataset.csv")
df_critical = select_target_columns(df)
df_cleaned = clean_critical_data(df_critical)

Dataset pulled successfully
Dataset cleaned succesfully
Rows before: 180519 | Rows after: 180519
Columns before: 9 | Columns after: 9
